In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind
from joblib import Parallel, delayed
import random
import time

# تنظیمات اولیه
random.seed(42)
np.random.seed(42)
lambda_penalty = 0.01

# بارگذاری و پیش‌پردازش داده‌ها
data = load_breast_cancer()
X, y = data.data, data.target
scaler = StandardScaler()
X = scaler.fit_transform(X)
n_features = X.shape[1]

# حافظه برای تابع برازندگی
fitness_cache = {}

# تابع برازندگی
def fitness(individual):
    ind_tuple = tuple(individual)
    if ind_tuple in fitness_cache:
        return fitness_cache[ind_tuple]
    selected_features = np.where(individual == 1)[0]
    if len(selected_features) == 0:
        fit = 0.0
    else:
        svm = SVC(kernel='linear', random_state=42)
        accuracy = cross_val_score(svm, X[:, selected_features], y, cv=5, scoring='accuracy').mean()
        penalty = lambda_penalty * (len(selected_features) / n_features)
        fit = accuracy - penalty
    fitness_cache[ind_tuple] = fit
    return fit

# الگوریتم ژنتیک بهینه‌شده
def genetic_algorithm(pop_size=50, n_generations=50, mutation_rate=0.01, crossover_rate=0.8, patience=5):
    population = [np.random.randint(0, 2, n_features) for _ in range(pop_size)]
    best_fitness = -np.inf
    best_individual = None
    no_improvement = 0
    
    for generation in range(n_generations):
        # موازی‌سازی محاسبه تابع برازندگی
        fitness_scores = Parallel(n_jobs=-1)(delayed(fitness)(ind) for ind in population)
        max_fitness = max(fitness_scores)
        mean_fitness = np.mean(fitness_scores)
        print(f"GA - نسل {generation+1}: بهترین برازندگی = {max_fitness:.4f}, میانگین برازندگی = {mean_fitness:.4f}")
        
        if max_fitness > best_fitness:
            best_fitness = max_fitness
            best_individual = population[np.argmax(fitness_scores)].copy()
            no_improvement = 0
        else:
            no_improvement += 1
        
        if no_improvement >= patience:
            print(f"توقف زودهنگام در نسل {generation+1}")
            break
        
        parents = []
        for _ in range(pop_size):
            tournament = random.sample(range(pop_size), 3)
            winner = tournament[np.argmax([fitness_scores[i] for i in tournament])]
            parents.append(population[winner].copy())
        
        new_population = []
        for i in range(0, pop_size, 2):
            if i + 1 < pop_size and random.random() < crossover_rate:
                crossover_point = random.randint(1, n_features - 1)
                parent1, parent2 = parents[i], parents[i + 1]
                child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
                child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
                new_population.extend([child1, child2])
            else:
                new_population.extend([parents[i].copy(), parents[i + 1].copy() if i + 1 < pop_size else parents[i].copy()])
        
        for ind in new_population:
            for j in range(n_features):
                if random.random() < mutation_rate:
                    ind[j] = 1 - ind[j]
        
        population = new_population[:pop_size - 1] + [best_individual.copy()]
    
    return best_individual, best_fitness

# الگوریتم میمتیک بهینه‌شده
def memetic_algorithm(pop_size=50, n_generations=50, mutation_rate=0.01, crossover_rate=0.8, ls_iterations=3, patience=5):
    population = [np.random.randint(0, 2, n_features) for _ in range(pop_size)]
    best_fitness = -np.inf
    best_individual = None
    no_improvement = 0
    
    for generation in range(n_generations):
        fitness_scores = Parallel(n_jobs=-1)(delayed(fitness)(ind) for ind in population)
        max_fitness = max(fitness_scores)
        mean_fitness = np.mean(fitness_scores)
        print(f"MA - نسل {generation+1}: بهترین برازندگی = {max_fitness:.4f}, میانگین برازندگی = {mean_fitness:.4f}")
        
        if max_fitness > best_fitness:
            best_fitness = max_fitness
            best_individual = population[np.argmax(fitness_scores)].copy()
            no_improvement = 0
        else:
            no_improvement += 1
        
        if no_improvement >= patience:
            print(f"توقف زودهنگام در نسل {generation+1}")
            break
        
        parents = []
        for _ in range(pop_size):
            tournament = random.sample(range(pop_size), 3)
            winner = tournament[np.argmax([fitness_scores[i] for i in tournament])]
            parents.append(population[winner].copy())
        
        new_population = []
        for i in range(0, pop_size, 2):
            if i + 1 < pop_size and random.random() < crossover_rate:
                crossover_point = random.randint(1, n_features - 1)
                parent1, parent2 = parents[i], parents[i + 1]
                child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
                child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
                new_population.extend([child1, child2])
            else:
                new_population.extend([parents[i].copy(), parents[i + 1].copy() if i + 1 < pop_size else parents[i].copy()])
        
        for ind in new_population:
            for j in range(n_features):
                if random.random() < mutation_rate:
                    ind[j] = 1 - ind[j]
        
        new_population = [local_search(ind, ls_iterations)[0] for ind in new_population]
        population = new_population[:pop_size - 1] + [best_individual.copy()]
    
    return best_individual, best_fitness

# جستجوی محلی
def local_search(individual, n_iterations=3):
    best_ind = individual.copy()
    best_fit = fitness(best_ind)
    
    for _ in range(n_iterations):
        new_ind = best_ind.copy()
        idx = random.randint(0, n_features - 1)
        new_ind[idx] = 1 - new_ind[idx]
        new_fit = fitness(new_ind)
        if new_fit > best_fit:
            best_ind = new_ind.copy()
            best_fit = new_fit
    
    return best_ind, best_fit

# اجرای الگوریتم‌ها
n_runs = 5  # کاهش تعداد اجرا برای سرعت بیشتر
ga_fitnesses = []
ma_fitnesses = []

for run in range(n_runs):
    print(f"\n--- اجرای {run+1} ---")
    start_time = time.time()
    ga_ind, ga_fit = genetic_algorithm()
    ga_time = time.time() - start_time
    ga_fitnesses.append(ga_fit)
    print(f"GA - زمان اجرا: {ga_time:.2f} ثانیه")
    
    start_time = time.time()
    ma_ind, ma_fit = memetic_algorithm()
    ma_time = time.time() - start_time
    ma_fitnesses.append(ma_fit)
    print(f"MA - زمان اجرا: {ma_time:.2f} ثانیه")

# مقایسه آماری
t_stat, p_value = ttest_ind(ga_fitnesses, ma_fitnesses)

# گزارش نتایج
print("\nنتایج الگوریتم ژنتیک (GA):")
print(f"میانگین برازندگی: {np.mean(ga_fitnesses):.4f} ± {np.std(ga_fitnesses):.4f}")
print(f"بهترین برازندگی: {max(ga_fitnesses):.4f}")

print("\nنتایج الگوریتم میمتیک (MA):")
print(f"میانگین برازندگی: {np.mean(ma_fitnesses):.4f} ± {np.std(ma_fitnesses):.4f}")
print(f"بهترین برازندگی: {max(ma_fitnesses):.4f}")

print("\nنتایج آزمون t-test:")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}")
if p_value < 0.05:
    print("تفاوت آماری معنادار است.")
else:
    print("تفاوت آماری معنادار نیست.")


--- اجرای 1 ---
GA - نسل 1: بهترین برازندگی = 0.9754, میانگین برازندگی = 0.9602
GA - نسل 2: بهترین برازندگی = 0.9754, میانگین برازندگی = 0.9632
GA - نسل 3: بهترین برازندگی = 0.9754, میانگین برازندگی = 0.9652
GA - نسل 4: بهترین برازندگی = 0.9774, میانگین برازندگی = 0.9683
GA - نسل 5: بهترین برازندگی = 0.9789, میانگین برازندگی = 0.9707
GA - نسل 6: بهترین برازندگی = 0.9789, میانگین برازندگی = 0.9724
GA - نسل 7: بهترین برازندگی = 0.9789, میانگین برازندگی = 0.9730
GA - نسل 8: بهترین برازندگی = 0.9789, میانگین برازندگی = 0.9749
GA - نسل 9: بهترین برازندگی = 0.9789, میانگین برازندگی = 0.9759
GA - نسل 10: بهترین برازندگی = 0.9792, میانگین برازندگی = 0.9758
GA - نسل 11: بهترین برازندگی = 0.9792, میانگین برازندگی = 0.9771
GA - نسل 12: بهترین برازندگی = 0.9803, میانگین برازندگی = 0.9773
GA - نسل 13: بهترین برازندگی = 0.9824, میانگین برازندگی = 0.9770
GA - نسل 14: بهترین برازندگی = 0.9824, میانگین برازندگی = 0.9768
GA - نسل 15: بهترین برازندگی = 0.9827, میانگین برازندگی = 0.9776
GA - نسل 16: بهتر